In [93]:
# Restaurant sentences 의 일부
sentences = [
"i want chinese food",
"i want thai food",
"i want italian food",
"i want a restaurant",
"i want to eat",
"i want to eat chinese food",
"i want to eat thai food",
"i am looking for chinese food",
"i am looking for a restaurant",
"can you find a chinese restaurant",
"can you find a thai restaurant",
"can you find an italian restaurant",
"tell me about chinese restaurants",
"tell me about thai restaurants",
"where can i eat chinese food",
]

In [94]:
# Tokenization
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

# Tokenized sentences를 사용해서 corpus를 생성
corpus = [tokenize(s) for s in sentences]

for sentence in corpus[:3]:
    print(sentence)

['<s>', 'i', 'want', 'chinese', 'food', '</s>']
['<s>', 'i', 'want', 'thai', 'food', '</s>']
['<s>', 'i', 'want', 'italian', 'food', '</s>']


In [95]:
# Unigram counts(코퍼스 내에서 특정 단어가 총 몇 번 등장 했는지) 계산
from collections import Counter

unigram_counts = Counter()

for sentence in corpus:
    unigram_counts.update(sentence)

# 10개의 가장 흔한 단어와 그 빈도를 출력    
print(unigram_counts.most_common(10))

# Unigram 확률 계산

# 단어가 나온 빈도의 합계를 계산
total_words = sum(unigram_counts.values())

def unigram_prob(word):
    # 단어의 unigram 확률을 계산 = (단어의 등장 횟수) / (전체 단어 수)
    return unigram_counts[word] / total_words

# want과 food의 unigram 확률을 계산
print(unigram_prob("i"))
print(unigram_prob("want"))
print(unigram_prob("food"))

[('<s>', 15), ('</s>', 15), ('i', 10), ('want', 7), ('food', 7), ('chinese', 6), ('restaurant', 5), ('thai', 4), ('a', 4), ('eat', 4)]
0.09259259259259259
0.06481481481481481
0.06481481481481481


In [96]:
# Generate restaurant sentences

import random

def sample_next_word_unigram(previous_word):
    candidates = []
    weights = []

    # unigram_counts에서 이전 단어(previous_word)와 함께 등장하는 단어들을 후보로 추가하고, 그 후보들의 count를 weights에 추가
    for w, count in unigram_counts.items():
        candidates.append(w)
        weights.append(count)

    # 후보 단어 중 하나를 샘플링하여 반환
    return random.choices(candidates, weights=weights, k=1)[0]

def generate_sentence(max_length = 15):
    output = []

    #    2. sample_next_word()를 사용하여 다음 단어를 샘플
    for _ in range(max_length):
        next_word = sample_next_word_unigram(unigram_counts)

        # 문장 끝
        if next_word == "</s>":
            break

        # 시작 토큰이 중간에 나오는 것을 제외!
        if next_word != "<s>":
            output.append(next_word)

    return " ".join(output)

for _ in range(10):
    print(generate_sentence())



tell i
find for eat can
want
find to can restaurant chinese food restaurants thai you

restaurants





In [97]:
# Perplexity Metric

# Perplexity는 언어 모델이 문장을 평가할 때 얼마나 헷갈려하는지를 나타내는 지표로, 낮을수록 모델이 문장을 잘 예측한다는 의미

import math

# Perplexity 계산
def perplexity_unigram(sentence):
    # sentence을 tokenize하기
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 1
    for i in range(n):
        # 토근화한 문장에서 unigram 확률을 계산
        p = unigram_prob(tokens[i])

        # 0으로 나누는 것을 방지하기 위해 조건문 추가
        if p == 0:
         return float("inf")

        # 부동소수점 연산에서 underflow를 방지하기 위해 log를 사용하여 확률을 더함
        log_prob += math.log(p)

    # log 계산 이후에 지수 함수 사용해서 원상 복구
    return math.exp(-log_prob / n)

test_sentences = [
"i want a chinese restaurant",
"where can i eat thai food",
"i love you!!!❤️", # unknown word -> inf 발생!
"my chickdee is so cute💕", # unknown word -> inf 발생!
"can you find thai food"
]

for s in test_sentences:
    print(s," : ",  perplexity_unigram(s))

i want a chinese restaurant  :  15.253236287845223
where can i eat thai food  :  22.06985976601905
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  21.718598173514383


In [98]:
# Bigram counts(코퍼스 내에서 특정 bigram이 총 몇 번 등장 했는지) 계산
bigram_counts = Counter()

for sentence in corpus:
    # bigram을 생성하고 bigram_counts에 추가
    # 단어 2개를 보기에 for 문의 범위를 len(sentence) - 1로 설정
    for i in range(len(sentence) - 1):
        # bigram 생성
        bigram = (sentence[i], sentence[i + 1])
        bigram_counts[bigram] += 1 # 바이그램이 등장할 때 마다 카운트 증가 -> 이 조합이 한번 더 나왔네!!

for bigram, count in bigram_counts.most_common(5):
    print(bigram, count)


# Bigram Probability 계산
def bigram_prob(previous_word, word):
    # bigram 확률 = P(word | previous_word) = count(previous_word, word) / count(previous_word) 

    # 이전 단어와 현재 단어의 bigram count
    numerator = bigram_counts[(previous_word, word)]

    # 이전 단어의 unigram count
    denominator = unigram_counts[previous_word]

    # 0으로 나누는 것을 방지하기 위해 조건문 추가
    if denominator == 0:
        return 0
    
    return numerator / denominator

print("\nBigram Probabilities:")
print("P(want | i) =", bigram_prob("i", "want"))
print("P(food | chinese) =", bigram_prob("chinese", "food"))
print("P(restaurant | a) =", bigram_prob("a", "restaurant"))

('<s>', 'i') 9
('i', 'want') 7
('food', '</s>') 7
('restaurant', '</s>') 5
('chinese', 'food') 4

Bigram Probabilities:
P(want | i) = 0.7
P(food | chinese) = 0.6666666666666666
P(restaurant | a) = 0.5


In [99]:
# 'Next Word' Prediction using bigram probabilities
def predict_bigram(previous_word, top_k=3):
    # 이전 단어(previous_word)와 함께 등장하는 단어들의 후보를 찾고, 그 후보들의 확률을 계산하여 상위 top_k개의 후보를 반환
    candidates = []

    for (w1, w2), count in bigram_counts.items():

        # 매개변수로 들어온 이전단어 (previous_word)와 일치하는 경우에만 후보로 추가
        if w1 == previous_word:

            # bigram 확률 계산하고
            probability = bigram_prob(w1, w2)

            # 후보에 넣기
            candidates.append((w2, probability))

    # 확률이 높은 순으로 정렬하고 상위 top_k개의 후보를 반환
    return sorted(candidates, key=lambda x: x[1], reverse=True)[:top_k]

# want 다음에 올 가능성이 가장 높은 상위 3개의 단어와 그 확률을 출력
predict_bigram("want")

[('to', 0.42857142857142855),
 ('chinese', 0.14285714285714285),
 ('thai', 0.14285714285714285)]

In [100]:
# Generate restaurant sentences

import random

def sample_next_word_bigram(previous_word):
    candidates = []
    weights = []
    # bigram_counts에서 이전 단어(previous_word)와 함께 등장하는 단어들을 후보로 추가하고, 그 후보들의 count를 weights에 추가
    for (w1, w2), count in bigram_counts.items():
        if w1 == previous_word:
            candidates.append(w2)
            weights.append(count)

    if not candidates:
        return "</s>"

    # 확률 기반 샘플링 -> 다음 단어를 고를 때 무조건 확률이 가장 높은 단어 하나만 고르는 것이 아니라, 확률에 따라 랜덤하게 선택
    return random.choices(candidates,weights=weights,k=1)[0]


def generate_sentence(max_length = 15):
    #    1. "<s>"로 시작
    current = "<s>"
    output = []

    #    2. sample_next_word()를 사용하여 다음 단어를 샘플
    for _ in range(max_length):
        next_word = sample_next_word_bigram(current)
        # 문장 끝
        if next_word == "</s>":
            break
        output.append(next_word)
        # 현재 단어 최신화
        current = next_word

    return " ".join(output)

for _ in range(10):
    print(generate_sentence())


i want a restaurant
can i want italian restaurant
can you find a restaurant
i want to eat thai food
i want a thai food
i eat thai food
where can i want a restaurant
i want thai restaurant
i want chinese food
i want to eat thai food


In [101]:
# Perplexity Metric

# Perplexity는 언어 모델이 문장을 평가할 때 얼마나 헷갈려하는지를 나타내는 지표로, 낮을수록 모델이 문장을 잘 예측한다는 의미

import math

# Perplexity 계산
def perplexity_bigram(sentence):
    # sentence을 tokenize하기
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 1
    for i in range(n):
        # 토근화한 문장에서 bigram 확률을 계산
        p = bigram_prob(tokens[i],tokens[i + 1])

        # 0으로 나누는 것을 방지하기 위해 조건문 추가
        if p == 0:
         return float("inf")

        # 부동소수점 연산에서 underflow를 방지하기 위해 log를 사용하여 확률을 더함
        log_prob += math.log(p)

    # log 계산 이후에 지수 함수 사용해서 원상 복구
    return math.exp(-log_prob / n)

test_sentences = [
"i want a chinese restaurant",
"where can i eat thai food",
"i love you!!!❤️", # unknown word -> inf 발생!
"my chickdee is so cute💕", # unknown word -> inf 발생!
"can you find thai food"
]

for s in test_sentences:
    print(s," : ",  perplexity_bigram(s))

i want a chinese restaurant  :  2.7144176165949068
where can i eat thai food  :  3.356538286432562
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  inf


In [102]:
# Trigram counts 구하기
trigram_counts = Counter()

for sentence in corpus:
    # Trigram을 고려하여 길이 조절
    for i in range(len(sentence) - 2):
            # 연속된 세 단어를 추출하여 trigram_counts에 추가
            w1, w2, w3 = sentence[i:i+3]

            # 연속된 3단어의 등장 횟수를 카운트하여 집계
            trigram_counts[(w1, w2, w3)] += 1


# Trigram Probability 계산
def trigram_prob(w1, w2, w3):
    numerator = trigram_counts[(w1, w2, w3)]
    denominator = bigram_counts[(w1, w2)]

    # 0으로 나누는 것을 방지하기 위해 조건문 추가
    if denominator == 0:
        return 0
    
    return numerator / denominator

# i와 want 다음에 chinese가 올 확률을 계산
print(trigram_prob("i", "want", "chinese"))

0.14285714285714285


In [103]:
# Generate restaurant sentences

import random

def sample_next_word_trigram(previous_word, previous_word2):
    # 이전 단어(previous_word)와 이전 이전 단어(previous_word2)를 고려하여 다음 단어를 샘플링 -> 트라이그램 모델에서는 이전 두 단어를 고려하여 다음 단어를 예측
    candidates = []
    weights = []
    # trigram_counts에서 이전 단어(previous_word)와 함께 등장하는 단어들을 후보로 추가하고, 그 후보들의 count를 weights에 추가
    for (w1, w2, w3), count in trigram_counts.items():
        if w1 == previous_word and w2 == previous_word2:
            candidates.append(w3)
            weights.append(count)

    if not candidates:
        return "</s>"

    # 확률 기반 샘플링 -> 다음 단어를 고를 때 무조건 확률이 가장 높은 단어 하나만 고르는 것이 아니라, 확률에 따라 랜덤하게 선택
    return random.choices(candidates,weights=weights,k=1)[0]


def generate_sentence_trigram(max_length = 15):
    #    1. "<s>"로 시작
    w1 = "<s>"
    w2 = "i" #가장 많이 나온 단어를 두 번째 단어로 설정

    # 생성된 단어들을 담을 리스트에 i를 추가해두기
    output = [w2]

    #    2. sample_next_word()를 사용하여 다음 단어를 샘플
    for _ in range(max_length):
        next_word = sample_next_word_trigram(w1, w2)

        # 문장 끝
        if next_word == "</s>":
            break

        output.append(next_word)
        # 현재 단어 최신화
        w1 = w2
        w2 = next_word

    return " ".join(output)

for _ in range(10):
    print(generate_sentence_trigram())


i want chinese food
i want chinese food
i want a restaurant
i want thai food
i want a restaurant
i am looking for a restaurant
i want to eat thai food
i am looking for chinese food
i want to eat
i want italian food


In [104]:
# Perplexity Metric

# Perplexity는 언어 모델이 문장을 평가할 때 얼마나 헷갈려하는지를 나타내는 지표로, 낮을수록 모델이 문장을 잘 예측한다는 의미

import math

# Perplexity 계산
def perplexity_trigram(sentence):
    # sentence을 tokenize하기
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 2
    for i in range(n):
        # 토근화한 문장에서 trigram 확률을 계산
        p = trigram_prob(tokens[i], tokens[i + 1], tokens[i + 2])

        # 0으로 나누는 것을 방지하기 위해 조건문 추가
        if p == 0:
            return float("inf")

        # 부동소수점 연산에서 underflow를 방지하기 위해 log를 사용하여 확률을 더함
        log_prob += math.log(p)

    # log 계산 이후에 지수 함수 사용해서 원상 복구
    return math.exp(-log_prob / n)

test_sentences = [
"i want a chinese restaurant",
"where can i eat thai food",
"i love you!!!❤️", # unknown word -> inf 발생!
"my chickdee is so cute💕", # unknown word -> inf 발생!
"can you find thai food"
]

for s in test_sentences:
    print(s," : ",  perplexity_trigram(s))

i want a chinese restaurant  :  inf
where can i eat thai food  :  inf
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  inf
